## Collection of ENSO phase specific diagnostics for use in papers and elsewhere
### - Plots linear rgression feedbacks for 2D fields
### - (Add feddbacks to 3D fields)
### - (Add RWS calculation).

In [5]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import cartopy.crs as ccrs

from scipy.stats import linregress
from windspharm.xarray import VectorWind

import importlib

import re
import os
os.cpu_count()

import enso_feedbacks_utils as mypy

In [6]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [7]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",   
    cores=4,
    memory="32GB",
    processes=4,      # one process per core (safe default)
    local_directory="/glade/derecho/scratch/rneale/dask-temp"  # <--- custom temp dir
)

cluster.scale(jobs=32)
client = Client(cluster)

client

## Setup Run Information

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Compute/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Compute/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.176:40149,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/Compute/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [8]:
''' CASE SPECIFICATIONS '''

#enso_cases = ['OBS','CESM2','CESM1','CESM3-156','CESM3-192']
#enso_names = [['ERA5','ERA5'],'b.e21.BHISTcmip6.f09_g17.LE2-1001.001','b.e11.B1850C5CN.f09_g16.005','b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156','b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156']

''' Sunset of cases? '''

loop_icases = 3  # Number of set cases to run.

#enso_cases = ['OBS','CESM2','CESM1']
#enso_names = [['ERA5','ERA5'],'b.e21.BHISTcmip6.f09_g17.LE2-1001.001','b.e11.B1850C5CN.f09_g16.005']

#enso_ystart = [1979,1850,410,31,31]
#enso_yend   = [1990,1899,470,70,70]
#enso_yend   = [2023,1899,490,70,70]


enso_cases = ['OBS','CESM2','CESM1']
enso_names = [['ERA5','ERA5'],'b.e21.BHISTsmbb.f09_g17.LE2-1011.001','b.e11.B20TRC5CNBDRD.f09_g16.001']

enso_ystart = [1979,1850,1850,31,31]
#enso_yend   = [1990,1870,1870,70,70]
enso_yend   = [2023,2005,2005,70,70]


#enso_cases = ['CESM2','CESM1']
#enso_names = ['b.e21.BHISTsmbb.f09_g17.LE2-1011.001','b.e11.B20TRC5CNBDRD.f09_g16.001']

#enso_ystart = [1850,1850,31,31]
#enso_yend   = [1990,1870,1870,70,70]
#enso_yend   = [2005,2005,70,70]


# F-case
#enso_cases = ['OBS','CAM6','CAM5']

#/glade/campaign/collections/gdex/data/d651010/tropical/f.e21.FHIST.f09_f09.historical.ersstv5.toga.ens01/atm/month_1
#enso_names = [['ERA5','ERA5'],'f.e21.FHIST_BGC.f09_f09.historical.ersstv5.goga.ens01','f.e11.FAMIPC5CN.f09_f09.historical.goga.ens01']

#enso_ystart = [1979,1888,1880]
#enso_yend   = [2023,2005,2005]

#enso_ystart = [1979,1979,1979]
#enso_yend   = [2005,2005,2005]


#enso_cases = ['CESM2']
#enso_names = ['b.e21.BHISTcmip6.f09_g17.LE2-1001.001']

#enso_ystart = [1850,430,31]
#enso_yend   = [1899,470,70]












#enso_cases = ['CESM2','CESM1']
#enso_names = ['b.e21.BHISTcmip6.f09_g17.LE2-1001.001','b.e11.B1850C5CN.f09_g16.005']


#enso_cases = ['OBS','CESM2','CESM1','156','162','163','166','170','171']
#enso_names = [['ERA5','TROPFLUX'], 
#              'b.e21.BHISTcmip6.f09_g17.LE2-1001.001',
#              'b.e11.B1850C5CN.f09_g16.005',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.162',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.163',
#              'b.e30_beta06.B1850C_LTso.ne30_t232_wgx3.166',
#              'b.e30_beta06.B1850C_LTso.ne30_t232_wgx3.170',
#              'b.e30_beta06.B1850C_LTso.ne30_t232_wgx3.171'
#             ]              

#enso_cases = ['ERA5','156']
#enso_names = ['ERA5',
#              'b.e30_alpha06e.B1850C_LTso.ne30_t232_wgx3.156']

fig_pref = 'test_f_20th'

xnino_reg = 'nino34' ; ynino_reg = 'nino34' ; 




#enso_ystart = [1979,1860,430,30,30,30,30,30,30]
#enso_yend   = [2018,1899,470,70,70,70,70,70,70]

#season = 'DJF' ; get_months = [12,1,2]
season = 'NDJFM' ; get_months = [11,12,1,2,3]
#season = 'ANN' ; get_months = [1,2,3,4,5,6,7,8,9,10,11,12]

# Select analysis

l1d_plot = False
l3d_plot = True
l3d_latlon = True
lrws_plot =  False



lread_in_all_hist = False   # Read in all the CESM/CAM history files in a directory (max out for 100 years)

lwrite_ts_file = True  # Write out a timeseries of 2D SST/VAR variables (if they don't exist)
lread_ts_file = True # Read in a timeseries of 2D SST/VAR variables (if they exist).


ncases = len(enso_cases)


is_in_situ = False  # For real, non-reanalysis observations.

## Grab data and averaged

In [9]:
''' VARIABLE SPECIFICATIONS '''

''' x-var '''
var_x = 'TS'  ; vunits_x = 'K' 
#var_x = 'PRECT' ; vunits_x = 'mm/day' # mon means (m->mm = *1000.)  
#var_y = 'TAUX'  ; vunits_y = 'N/m^2'
#var_comp = 'taux' ; evar_comp = 'chnk' ; ovar_scale = -30.*1.e3 ; cvar_comp = 'TAUX' ; cvar_comp = 'TAUX'; cvar_scale = -1.*1.e3 ;  vunits = 'N/m^2'

''' y-var '''
#var_y = 'OMEGA500'  ; vunits_y = 'mb/hr'
#var_y = 'DIV200'  ; vunits = '/s*'
#var_y = 'PRECT' ; vunits_y = 'mm/day' # mon means (m->mm = *1000.)  
var_y = 'TAUX' ; vunits_y = 'Nm$^{-2}$'
#var_y = 'DTCOND300'  ; vunits_y = 'K/day'

''' 3D '''
var_y3d = 'OMEGA' ;  vunits_y3d='mb/hr'  ; xlim_pdf = [-1,1] ; crange = [-2.,2.]
#var_y3d = 'Q' ;  vunits_y3d='g/kg'  ; xlim_pdf = [-2.,2] ; crange = [-3.,3.]
#var_y3d = 'T' ;  vunits_y3d='K'  ; xlim_pdf = [-2.,2] ; crange = [-3.,3.]
#var_y3d = 'CLOUD' ;  vunits_y3d='%'  ; xlim_pdf = [-2.,2] ; crange = [-25.,25.]
#var_y3d = 'U' ;  vunits_y3d='%'  ; xlim_pdf = [-2.,2] ; crange = [-10.,10.]
#var_y3d = 'RELHUM' ;  vunits_y3d='%'  ; xlim_pdf = [-2.,2] ; crange = [-25.,25.]
#var_y3d = 'DTCOND' ;  vunits_y3d='K/day'  ; xlim_pdf = [-2.,2] ; crange = [-5.,5.]
#var_y3d = 'DCQ' ;  vunits_y3d='g/kg/day'  ; xlim_pdf = [-2.,2] ; crange = [-2,2]

#var_y3d = 'DIV' ;  vunits_y3d='s$^{-1}$*1e$^{6}$'  ; xlim_pdf = [-5.,5] ; crange = [-3.,3.]

#var_y3d = 'RELHUM' ; '%'

''' Lat Lon '''
var_latlon_3d = 'Z3' ; var_latlon_p = 500; vunits_ll = 'm' ; pregion='NPac'
#var_latlon_3d = 'V' ; var_latlon_p = 850; vunits_ll = 'm/s' ; pregion='NPac'

# Get correct variable and remove numbers from Z3

var_no_num = re.sub(r"\d+", "", var_latlon_3d)
var_latlon_2d = f"{var_no_num}{int(var_latlon_p)}"




In [10]:
importlib.reload(mypy)

### CHANGE for your own local figure output location
dir_fig = '/glade/u/home/rneale/python/python-figs/papers/ENSO_vprocs_fbacks/'

fscale = 1.0 # Hopefully scales all the text when I chnage figure size.

figy = 12
figx = 12

#plt.rcParams.update({
#            "font.size": 12 * fscale,
#            "axes.titlesize": 14 * fscale,
#            "axes.labelsize": 12 * fscale,
#            "xtick.labelsize": 10 * fscale,
#            "ytick.labelsize": 10 * fscale,
#            "legend.fontsize": 10 * fscale,
#        })


### Settings

if l1d_plot:
    figp, axp = plt.subplots(
        nrows=1,
        ncols=2,
        constrained_layout=True,
        figsize=(22, 10),
        )
    
    figs, axs = plt.subplots(
        nrows=1,
        ncols=1,
        figsize=(figy, figx),
        )


#if l3d_latlon:##

#    ccrs_preg = ccrs.Robinson(central_longitude=180.) if pregion == 'Global' else ccrs.PlateCarree(central_longitude=180.)
    
#    fig_ll, ax_ll = plt.subplots(
#        nrows=1,
#        ncols=1,
#        figsize=(12, 12),
#        subplot_kw={'projection': ccrs_preg}, constrained_layout=True
#        )


plt.rcdefaults()
plt.rcParams.update({'font.size': 18*fscale})


# Plot ranges

xmin,xmax,x_avals = mypy.fig_domains(var_x)
ymin,ymax,y_avals = mypy.fig_domains(var_y)



scat_cols = [
    "black",
    "red",
    "royalblue",
    "darkorange",
    "forestgreen",
    "firebrick",
    "goldenrod",
    "mediumpurple",
    "deepskyblue",
    "crimson"
    
]


# Analysis summary

print('### Analyzing ',var_y,' nino anomaly dependence on ',var_x,' nino anomalies') 
print('### Analyzing ',var_y3d,' nino anomaly dependence on ',var_x,' nino anomalies') 





''''''
''' LOOP OVER CASES '''
''''''

for icase,case in enumerate(enso_cases[0:loop_icases]):


    lstyle = '--' if icase==0  else '-'
    lstyle = '-.' if icase in [1,2]  else '-'
    
    lmark = '^' if icase==0 else ''
        
    
    cname = enso_names[icase]

    
# Specifies the case as a single string (deals with multiple entries for obs. source).

    fcname = cname[0] if (isinstance(cname, list)) else cname


    
    yr0 = enso_ystart[icase] 
    yr1 = enso_yend[icase] 


    print('')
    print([icase+1],' of ',[ncases],' +++ ' , enso_names[icase] , ' +++')
    print('-- Reading in data --')


   
    
    # Case 'type'

    ctype = 'OBS'
    if 'b.e3' in cname : ctype = 'cesm3'
    if 'b.e2' in cname : ctype = 'cesm2'
    if 'b.e1' in cname : ctype = 'cesm1'
    if 'f.e1' in cname : ctype = 'cam5'
    if 'f.e2' in cname : ctype = 'cam6'

    


    '''
        Start 1D Anlaysis
    '''
    
    # 1. Grab data for x-axis and y-axis variable (obs. or model)

    print('')
    print('** 1. Grabbing Case Data')

    # Some manipulation because of obs. could be different types ('cases').
    cname_xy = cname
    if not isinstance(cname, list): cname_xy = [cname]
        

# Grab variable for independent (z) and dependent (y) axes.
 
    da_x = mypy.get_dataset(cname_xy[0],ctype,var_x,yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)
    da_y = mypy.get_dataset(cname_xy[-1],ctype,var_y,yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)

# 2. Time info check

    print('')
    print('** 2. Checking and Trimming Time Dimensions')
    da_x, da_y = mypy.dataset_ts_trim(da_x, da_y, yr0, yr1, var_x, var_y, get_months)


    
    # Calculate nino3.4 and VAR timeseries.
    print('')
    print('** 3. Calculating 1D Nino Anomalies')

    
    var_x_1d, var_x_pdf = mypy.nino_anom_ts(da_x,xnino_reg,x_avals) # This will be used for the 3D field figures.
    var_y_1d, var_y_pdf = mypy.nino_anom_ts(da_y,ynino_reg,y_avals)
    
    var_x_1d = var_x_1d.compute()
    var_y_1d = var_y_1d.compute()

   
    
    ''' 
        Scatter plot of the relationship between SST and VAR 
    '''    
    
    # Linear regression

    if l1d_plot:

        print('')
        print('** 4. Plotting 1D PDFs')
    
    
     
        
        # Histogram
        hist, edges = np.histogram(var_y_1d, bins=30, density=True)
        area_hist = np.sum(hist * np.diff(edges))
        
      
    
        axp[0].hist(var_x_1d, linewidth=5, color=scat_cols[icase],  bins =  int(0.5*var_x_pdf.size),range=(xmin, xmax), alpha = 0.3, density=True)
        axp[0].plot(x_avals, var_x_pdf, linewidth=5, color=scat_cols[icase],linestyle=lstyle, label=case)
        
        axp[1].hist(var_y_1d, linewidth=5, color=scat_cols[icase],  bins =  int(0.5*var_y_pdf.size),range=(ymin, ymax),alpha = 0.3, density=True)
        axp[1].plot(y_avals, var_y_pdf, linewidth=5, color=scat_cols[icase],linestyle=lstyle, label=case)
        
        
        axp[0].set_xlim([xmin,xmax]) ; axp[1].set_xlim([ymin,ymax])
    
        
        axp[0].set_title(var_x+' PDF ('+xnino_reg+') - '+season) ; axp[1].set_title(var_y+' PDF ('+ynino_reg+') - '+season) 
        axp[0].set_xlabel(vunits_x) ; axp[1].set_xlabel(vunits_y) 
        axp[0].set_ylabel('Density') ; axp[1].set_ylabel('Density')
        axp[0].grid(True) ; axp[1].grid(True) 
        axp[0].axvline(x=0., color='gray',linestyle='--', linewidth=2.5) ; axp[1].axvline(x=0., color='gray',linestyle='--', linewidth=2.5)
        axp[0].legend() ; axp[1].legend() 
     
    
    
        
        '''
           LAG CORRELATIONS
        '''
        
        
        print('')
        print('** 5. Calculating and Plotting 1D Lag Regress')
    
        # Lag regress.
    
        
#        slope_neg, intercept_neg, r_value_neg, p_value_neg, std_err_neg = linregress(var_x_1d[var_x_1d < 0], var_y_1d[var_x_1d < 0])
#        slope_pos, intercept_pos, r_value_pos, p_value_pos, std_err_pos = linregress(var_x_1d[var_x_1d > 0], var_y_1d[var_x_1d > 0])
    
       
        slope, intercept, r_value, p_value, std_err = linregress(var_x_1d, var_y_1d)
     
    
        # Constructing linear lines to plot.
        varx_even_neg = np.linspace(xmin, 0, 50) 
        line_neg = slope_neg * varx_even_neg + intercept_neg
    
        varx_even_pos = np.linspace(0, xmax, 50) 
        line_pos = slope_pos * varx_even_pos + intercept_pos
    
        varx_even = np.linspace(xmin, xmax, 100) 
        line = slope * varx_even + intercept
        
        
        # Scatter plots.
        
        case_yrs = case+' ['+str(int(yr0))+'-'+str(int(yr1))+'] '+f"{slope:.3g}"
        axs.scatter(var_x_1d, var_y_1d, label=case_yrs, alpha=0.2, color=scat_cols[icase],s=12)
    
        # Lag regress line on top of scatter
        axs.plot(varx_even, line, color=scat_cols[icase],linewidth=4,linestyle=lstyle)
#        axs.plot(varx_even_neg, line_neg, color=scat_cols[icase],linewidth=4,linestyle=lstyle)
#        axs.plot(varx_even_pos, line_pos, color=scat_cols[icase],linewidth=4,linestyle=lstyle)
        
        axs.set_xlabel(var_x+' ('+vunits_x+')') ; axs.set_xlim([xmin,xmax])
        axs.set_ylabel(var_y+' ('+vunits_y+')') ; axs.set_ylim([ymin,ymax])
        axs.set_title('Linear Regression '+var_x+' ('+xnino_reg+') with '+var_y+' ('+ynino_reg+') - '+season)
    
        # -ve and +ve dependency selection?
        
        if icase==ncases-1:
            legend = axs.legend(title='Case/Years/Slope')
            legend_labels = legend.get_texts()
            legend.get_title().set_fontweight('bold') 
            for ileg, leg_lab in enumerate(legend_labels): leg_lab.set_color(scat_cols[ileg])   # Dataset 1
            
        axs.grid(True)
        axs.axhline(0, color='gray', linestyle='--', linewidth=2.5)
        axs.axvline(0, color='gray', linestyle='--', linewidth=2.5)

         







    '''
        Start 3D Analysis
    '''
    

    if l3d_plot:
    
        print('')
        print('** 6. 3D: Reading Data')
    


        fig_zp, ax_zp = plt.subplots(
            nrows=1,
            ncols=1,
            figsize=(figy, figx),
            )

        
    
        fig_zm, ax_zm = plt.subplots(
            nrows=1,
            ncols=1,
            figsize=(figy, figx),
            )


        
        # Grab a 3D variable to be plot on the y-axis.
      
        if var_y3d == "DIV" and case != 'OBS':

            print(' -Divergence from CESM U and V')
            
            da_u3d = mypy.get_dataset(cname_xy[-1],ctype,'U',yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)
            da_v3d = mypy.get_dataset(cname_xy[-1],ctype,'V',yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)

          
            erad = 6.371e6  # Earth radius (m)


            lat = np.deg2rad(da_u3d.lat)
            coslat = np.cos(lat).where(np.abs(np.cos(lat)) > 1e-3)
            
            dlon = np.deg2rad(da_u3d.lon.diff("lon").mean())
            dlat = np.deg2rad(da_u3d.lat.diff("lat").mean())
            
            dudlon = da_u3d.diff("lon") / dlon
            dvdlat = da_v3d.diff("lat") / dlat
            
            da_y3d = dudlon / (erad * coslat) + dvdlat / erad
            da_y3d = da_y3d * 1e6


 
        else:
            
            da_y3d = mypy.get_dataset(cname_xy[-1],ctype,var_y3d,yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)

     
        
        # 2. Time info check
    
        print('')
        print('** 7. 3D: Checking and Trimming Time Dimensions (3d only)')
        _, da_y3d = mypy.dataset_ts_trim(da_x, da_y3d, yr0, yr1, var_x, var_y, get_months)

        
        # Calculate nino3.4 and VAR timeseries.
        print('')
        print('** 8. 3D: Calculating 3D Nino Anomalies (already have 1D x-axis)')
    
     
        var_y_3d, var_y_pdf3d = mypy.nino_anom_ts(da_y3d,ynino_reg,y_avals)
    
       
    
        print('** 9. 3D: Plot PDFs: Overlayed for each pressure level.')
        print('')
    
    
    # Define two colors
        color1 = "green"
        color2 = "red"
    
    # Number of levels
        nlevels = 12
    
    # Create a colormap that linearly interpolates between color1 and color2
 
        full_cmap = plt.get_cmap('gist_rainbow_r') 
    
    # Reduce to 10 discrete colors
        cmap = full_cmap(np.linspace(0, 1, nlevels))  # array of RGBA colors
        
    # Example data
        col_plev = np.linspace(0, 1, nlevels).reshape(1, -1)
    
        plev_range = [100.,1000.]
    
        plev_plot = (var_y_pdf3d.plev > plev_range[0]) & (var_y_pdf3d.plev < plev_range[1])
    
       
        for il, lev in enumerate(var_y_pdf3d.plev.where(plev_plot,drop=True).values):
            ax_zp.plot(
                var_y_pdf3d.axis,
                var_y_pdf3d.sel(plev=lev),
                label=f"{int(lev)} hPa"
        )
    
        ax_zp.text(
            0.01, 0.99, fcname+' ('+str(yr0)+'-'+str(yr1)+')',   # y < 0 puts it below xlabel
            transform=ax_zm.transAxes,
            ha="left",
            va="top",
            fontweight="bold",
            fontsize=10*fscale
        )

#        ptitle2 = r"$\bf{"+case+r"}$ - " + var_y3d + " " + ynino_reg + " anomalies by pressure"
        
        ax_zp.set_title(r"$\bf{"+case+"}$ - "+var_y3d+" PDF ("+ynino_reg+")") 
        ax_zp.set_xlabel(vunits_y3d) 
        ax_zp.set_ylabel('Density')
        ax_zp.set_xlim(xlim_pdf) 
        ax_zp.grid(True) 
        ax_zp.axvline(x=0., color='gray',linestyle='--', linewidth=2.5)
        ax_zp.legend() 
    
    
    
    
    
    
    
    
    
        '''
            3D: Mean of Points With Height'
        '''
        
    
        
        print('** 10. 3D: Mean of Points With Height')
        print('')
    
        nlevels = 20 # Bins and contours.
    
    # bin levels
        dlev_b = ((xmax-xmin)/nlevels)
        
        bins = np.arange(xmin, xmax+dlev_b, dlev_b)     # anomaly bins (K)
        bin_centers = 0.5 * (bins[:-1] + bins[1:]) 
        binned, binned_var = mypy.bin_mean_var_by_level(var_y_3d, var_x_1d, bins,  bin_centers)

        
        
    # Contour levels
        dlev_p = ((crange[1]-crange[0])/nlevels)
        levels = np.arange(crange[0],crange[1]+dlev_p,dlev_p)
        slevels = np.array([0.1,0.2,0.4,1.,1.5,2.])*1.
        swidths = np.array([1,1,1,2,2,2])
        
        
        cf = ax_zm.contourf(
            binned.bin,
            binned.plev,
            binned,
            levels=levels,
            cmap="RdBu_r",
            extend="both"
        )
    
        plt.colorbar(cf, ax=ax_zm, label="Anomaly ("+vunits_y3d+")")
    
        
        cf = ax_zm.contour(
            binned.bin,
            binned.plev,
            binned_var,
            levels=slevels,
            linewidths = swidths,
            colors='black'
        )
        ax_zm.clabel(cf,fontsize=12)
        
        ax_zm.invert_yaxis()
        ax_zm.set_xlim([xmin,xmax]) 

        btitle1 = f"{xnino_reg} {var_x} anomaly ({vunits_x})"
#        btitle1 = (
#            f"{xnino_reg} {var_x} anomaly ({vunits_x})\n"
#            f"<span style='font-weight:bold; font-size:0.85em'>{cname}</span>"
#        )
        
        ax_zm.set_xlabel(btitle1,usetex=False)
        ax_zm.set_ylabel("Pressure (hPa)")

        ax_zm.text(
            0.01, 0.99, fcname+' ('+str(yr0)+'-'+str(yr1)+')',   # y < 0 puts it below xlabel
            transform=ax_zm.transAxes,
            ha="left",
            va="top",
            fontweight="bold",
            fontsize=10*fscale
        )

        btitle2 = r"$\bf{"+case+r"}$ - " + var_y3d + " " + ynino_reg + " anomalies by pressure - "+season

        
        ax_zm.set_title(btitle2,usetex=False)
        ax_zm.axvline(x=0., color='gray',linestyle='--', linewidth=2.5)
    
        
        fig_zp.savefig(dir_fig+fcname+'_'+var_x+'_'+var_y3d+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_plev_PDF_monthly.png')
        fig_zm.savefig(dir_fig+fcname+'_'+var_x+'_'+var_y3d+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_plev_monthly.png')
        
        fig_zp.show()
        fig_zm.show()
        
        plt.close(fig_zp)
        plt.close(fig_zm)
        

    '''
       2D: Lat-Lon Plot'
    '''


    if l3d_latlon:  

        print('')
        print('** 11. 2D lat-lon: Checking and Trimming Time Dimensions (3d only)')

   
        

        ccrs_preg = ccrs.Robinson(central_longitude=180.) if pregion == 'Global' else ccrs.PlateCarree(central_longitude=180.)
    
        fig_llo, ax_llo = plt.subplots(
            nrows=1,
            ncols=1,
            figsize=(figy, figx),
            subplot_kw={'projection': ccrs_preg}, constrained_layout=True
        )

        fig_lla, ax_lla = plt.subplots(
            nrows=1,
            ncols=1,
            figsize=(figy, figx),
            subplot_kw={'projection': ccrs_preg}, constrained_layout=True
        )


 
        
         # Grab a 3D variable to be plot on the y-axis.

        if var_latlon_3d == "DIV" and case != 'OBS':
            
            
            da_u3d = mypy.get_dataset(cname_xy[-1],ctype,'U',yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)
            da_v3d = mypy.get_dataset(cname_xy[-1],ctype,'V',yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)

          
            erad = 6.371e6  # Earth radius (m)

            lat = np.deg2rad(da_u3d.lat)
            lon = np.deg2rad(da_u3d.lon)

            coslat = np.cos(lat)

            dudlon = da_u3d.differentiate("lon") * np.pi/180
            dvdlat = da_v3d.differentiate("lat") * np.pi/180

            da_y3d = (dudlon + dvdlat) / (erad * coslat)

            

        else:
            da_y3d = mypy.get_dataset(cname_xy[-1],ctype,var_latlon_3d,yr0,yr1,lread_in_all_hist,lwrite_ts_file,lread_ts_file)


        
        # Select pressure level.
        plot_latlon = da_y3d.sel(plev=var_latlon_p).squeeze(drop=True)

        
        # 2. Time info check
    
        print('')
        print('** 12. 2D lat-lon: Checking and Trimming Time Dimensions (3d only)')
        _, plot_latlon = mypy.dataset_ts_trim(da_x, plot_latlon, yr0, yr1, var_x, var_y, get_months)
    
        
        # Calculate nino3.4 and VAR timeseries.
        print('')
        print('** 13. 2D lat-lon: Create Monthly mean El nino/La Nina anomlaies and climo from 2D lat_lon field')
    
        plot_latlon, var_x_1d = xr.align(plot_latlon, var_x_1d)
        
        plot_latlon_clim = plot_latlon.groupby("time.month").mean("time", keep_attrs=True)
        plot_latlon_anom = plot_latlon.groupby("time.month") - plot_latlon_clim
        
        
        
        print('** 14. 2D lat-lon: Plot PDFs: Overlayed for each pressure level.')
        print('')

        

        
        # Plot Nino composite 


        nino_mask        = (plot_latlon_anom.time.month.isin(get_months)) & (var_x_1d >= 1.0)
        plot_latlon_nino = plot_latlon_anom.sel(time=nino_mask).mean("time")

       
        mypy.plot_latlon(ax_llo,fig_llo,icase,var_latlon_2d,plot_latlon_nino,vunits_ll,case,fcname,season,pregion,ynino_reg,False,False,False,fscale)

                 
        fig_llo.savefig(dir_fig+fcname+'_'+var_x+'_'+var_latlon_2d+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_latlon_nino_monthly.png')


        # Plot Nina composite 

        nina_mask        = (plot_latlon_anom.time.month.isin(get_months)) & (var_x_1d <= 1.0)
        plot_latlon_nina = plot_latlon_anom.sel(time=nina_mask).mean("time")

       
        mypy.plot_latlon(ax_lla,fig_lla,icase,var_latlon_2d,plot_latlon_nina,vunits_ll,case,fcname,season,pregion,ynino_reg,False,False,False,fscale)

                 
        fig_lla.savefig(dir_fig+fcname+'_'+var_x+'_'+var_latlon_2d+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_latlon_nina_monthly.png')


        fig_llo.show()      
        fig_lla.show()
                        
        plt.close(fig_llo)
        plt.close(fig_lla)

    if lrws_plot:
        u = ds["U"].sel(lev=200)
        v = ds["V"].sel(lev=200)
       

        w = VectorWind(u, v)

        div = w.divergence()
        vort = w.vorticity()
  




# OUTPUT

plt.show()

if l1d_plot:
    figp.savefig(dir_fig+fig_pref+'_'+var_x+'_'+var_y+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_1D_PDFs_monthly.png')
    figs.savefig(dir_fig+fig_pref+'_'+var_x+'_'+var_y+'_'+season+'_'+xnino_reg+'_'+ynino_reg+'_scatter_monthly.png')

print('+++++++++++++++++++++++++++++++++')
print('           -- DONE --')
print('+++++++++++++++++++++++++++++++++')


### Analyzing  TAUX  nino anomaly dependence on  TS  nino anomalies
### Analyzing  OMEGA  nino anomaly dependence on  TS  nino anomalies

[1]  of  [3]  +++  ['ERA5', 'ERA5']  +++
-- Reading in data --

** 1. Grabbing Case Data
  - Grabbing  ERA5  data for TS
  - Grabbing  ERA5  data for TAUX

** 2. Checking and Trimming Time Dimensions
  - Requested variable year range =  1979 - 2023
  - Time details for x-axis - TS
    - Available year range =  1978.Jan - 2023.Dec
  - Time details for y-axis - TAUX
    - Available year range =  1979.Jan - 2023.Dec

** 3. Calculating 1D Nino Anomalies

** 6. 3D: Reading Data
  - Grabbing  ERA5  data for OMEGA

** 7. 3D: Checking and Trimming Time Dimensions (3d only)
  - Requested variable year range =  1979 - 2023
  - Time details for x-axis - TS
    - Available year range =  1979.Jan - 2023.Dec
  - Time details for y-axis - TAUX
    - Available year range =  1979.Jan - 2023.Dec

** 8. 3D: Calculating 3D Nino Anomalies (already have 1D x-axis)
** 9. 3

/glade/work/rneale/conda-envs/neale_vproc2/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,



** 11. 2D lat-lon: Checking and Trimming Time Dimensions (3d only)
  - Grabbing  ERA5  data for Z3

** 12. 2D lat-lon: Checking and Trimming Time Dimensions (3d only)
  - Requested variable year range =  1979 - 2023
  - Time details for x-axis - TS
    - Available year range =  1979.Jan - 2023.Dec
  - Time details for y-axis - TAUX
    - Available year range =  1979.Jan - 2023.Dec

** 13. 2D lat-lon: Create Monthly mean El nino/La Nina anomlaies and climo from 2D lat_lon field
** 14. 2D lat-lon: Plot PDFs: Overlayed for each pressure level.


[2]  of  [3]  +++  b.e21.BHISTsmbb.f09_g17.LE2-1011.001  +++
-- Reading in data --

** 1. Grabbing Case Data
  - Grabbing file(s) for LENS1/2 (CESM1/2)
  - Grabbing file(s) for LENS1/2 (CESM1/2)

** 2. Checking and Trimming Time Dimensions
  - Requested variable year range =  1850 - 2005
  - Time details for x-axis - TS
    - Available year range =  1850.Feb - 2015.Jan
  - Time details for y-axis - TAUX
    - Available year range =  1850.Feb - 20

/glade/work/rneale/conda-envs/neale_vproc2/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,



** 11. 2D lat-lon: Checking and Trimming Time Dimensions (3d only)
  - Grabbing file(s) for LENS1/2 (CESM1/2)
-Interpolating  Z3  to  [100000.  92500.  85000.  70000.  60000.  50000.  40000.  30000.  25000.
  20000.  15000.  10000.   7000.   5000.   3000.   2000.]  mb
Done

** 12. 2D lat-lon: Checking and Trimming Time Dimensions (3d only)
  - Requested variable year range =  1850 - 2005
  - Time details for x-axis - TS
    - Available year range =  1850.Feb - 2005.Dec
  - Time details for y-axis - TAUX
    - Available year range =  1850.Feb - 2006.Jan

** 13. 2D lat-lon: Create Monthly mean El nino/La Nina anomlaies and climo from 2D lat_lon field
** 14. 2D lat-lon: Plot PDFs: Overlayed for each pressure level.

+++++++++++++++++++++++++++++++++
           -- DONE --
+++++++++++++++++++++++++++++++++
